*Title:* 3_PCA
*Goal:* Generate environmental and outlier (based on allele freq. diff) PCAs

From original scripts: PCA_env and PCA_outliers

# 3.1_Environmental PCA
Data Manipulation  ----------------------
``` R
library(ggplot2)
library(tidyverse)
env_data <- read.csv("Stickleback_Data_Field_Data_2016_2017.csv") 
env_data<-env_data[c(1:271),] #removing empty rows

#Make a year column:
env_data$Year <- as.numeric(paste0("20", substr(env_data$Date, nchar(env_data$Date) - 1, nchar(env_data$Date))))

#Renaming and later removing LAguna Back Pond, as well as other sites I don't use 
env_data <- env_data %>%
  mutate(Code = recode(Site,
  `Laguna Creek Back Pond` = "LBP",
  "Younger Lagoon"="Younger",
  "Old Dairy Creek"="OldDairy",
  "Laguna Creek"="Laguna",
  "Lombardi Creek"= "Lombardi",
  "Whitehouse Creek"="WHT",
"Scott Creek "="Scott",
"Waddell Creek"="Waddell",
"Scott Creek"= "Scott")) %>%
filter(!Code %in% c("LBP","WHT")) #removing LBP and WHT from the dataset


#I want to see the change from Spring-Fall within each Code and Year:
seasonal_change <- env_data %>%
  group_by(Code, Year) %>%
  summarise(
    DO_change = mean(DO.mg.L.[Semiannual_Season == "Fall"], na.rm = TRUE) - 
                mean(DO.mg.L.[Semiannual_Season == "Spring"], na.rm = TRUE),
    Salinity_change = mean(Salinity..ppt.[Semiannual_Season == "Fall"], na.rm = TRUE) - 
                      mean(Salinity..ppt.[Semiannual_Season == "Spring"], na.rm = TRUE),
    Temp_change = mean(Temp..degrees.C.[Semiannual_Season == "Fall"], na.rm = TRUE) - 
                  mean(Temp..degrees.C.[Semiannual_Season == "Spring"], na.rm = TRUE)
  ) %>%
  as.data.frame()


#Making a column that combines site_Season_Year
env_data$Sample <- paste(env_data$Code, env_data$Semiannual_Season, env_data$Year, sep = "_")
#Select relevant columns and removing estuaries I don't care about (i.e. LAG bak pond and White):
env<-env_data[,c(44,13,15:17)] 

#I want to calculate the means, n, and sd for each variable, grouping by Sample:
summary_env <- env %>%
  group_by(Sample) %>%  # Group by the first column
  summarise(
    across(c("DO.mg.L.","Salinity..ppt.","Temp..degrees.C.","pH"),  # Apply functions to columns 2 through 5
           list(mean = ~mean(.x, na.rm = TRUE), 
                sd = ~sd(.x, na.rm = TRUE), 
                n = ~sum(!is.na(.x))),
           .names = "{.col}_{.fn}")
  )  %>%
  as.data.frame() 

rownames(summary_env) <- summary_env$Sample  # Set the row names to the Sample column (to preserve names after PCA)
summary_env  <- summary_env[, -1]

#Also, will only run the PCA on the means.
pca_data<-summary_env[,c(1,4,7)]

#now splitting the data into 2 separate years:
pca_data_2016<-pca_data %>%
  filter(grepl("2016", rownames(pca_data))) 

pca_data_2017<-pca_data %>%
  filter(grepl("2017", rownames(pca_data))) 

# Generate PCA ----------------------
#centering and scaling (on means)
pca_result_2016 <- prcomp(pca_data_2016, center = TRUE, scale. = TRUE)
names(pca_result_2016)
pca_result_2017 <- prcomp(pca_data_2017, center = TRUE, scale. = TRUE)

# Summary of PCA results
summary(pca_result_2016) #note that the $Proportion of Variance shows the variance explained by each PC.
summary(pca_result_2017) #note that the $Proportion of Variance shows the variance explained by each PC.

# Graph PCA ----------------------
library(ggplot2)
library(factoextra)
library(tidyverse)
library(dplyr)
library(data.table)
library(patchwork)
library(ggtext)

#Scree plot:
scree_plot_2016 <- fviz_eig(pca_result_2016)
scree_plot_2017 <- fviz_eig(pca_result_2017)
ggsave(filename = "scree_plot_pca_env_2016.png", plot = scree_plot_2016, width = 8, height = 6, dpi = 300)
ggsave(filename = "scree_plot_pca_env_2017.png", plot = scree_plot_2017, width = 8, height = 6, dpi = 300)


#process_pca <- function(pca_result, year) {
#Variance Explained:
result_17<-pca_result_2017 #manually change for either year 
explained_variance_17 <- result_17$sdev^2 / sum(result_17$sdev^2) * 100
pc1_var_17 <- round(explained_variance_17[1], 2)
pc2_var_17 <- round(explained_variance_17[2], 2)

#Label the points the same way as the SNP PCA:
result_17$Names<-rownames(result_17$x)
result_17$Estuary<- sub("_.*", "", result_17$Names)
result_17$Season <- sub("^[^_]*_", "", result_17$Names)

#PReparing for customized version:
pc_scores_17 <- as.data.frame(result_17$x)  # Principal component scores
pc_scores_17$Estuary <- result_17$Estuary  # Add Estuary grouping
pc_scores_17$Season <- str_extract(result_17$Season, "[A-Za-z]+") #redoing to only get the seaosn, not also the year. Also doing this because fill is messed up.
pc_scores_17$Season <- as.factor(pc_scores_17$Season)

pc_loadings_17 <- as.data.frame(result_17$rotation)  # Variable loadings
pc_loadings_17$Variable <- c("Dissolved Oxygen","Salinity","Temperature") # Add variable names

pc_loadings_17 <- pc_loadings_17 %>%
  mutate(
    vjust = ifelse(PC2 > 0, -1.5, 1.5),  # Adjust vertical position based on arrow direction
    hjust = ifelse(PC1 > 0, 0.3, 0.7)    # Adjust horizontal position based on arrow direction
  ) 

pca_colored_plot_17 <- ggplot(pc_scores_17, aes(x = PC1, y = PC2, color = Estuary, shape = Season)) +
  geom_point(size = 2, stroke = 1.2) +
  scale_color_manual(values = c("Younger" = "#2E2585", "Scott" = "#34B6FF", "Lombardi" = "#EADA02", "OldDairy" = "#D55E00", "Laguna" = "#287762", "Waddell" = "#A000E0")) +
  scale_shape_manual(values = c("Spring" = 1, "Fall" = 19)) + # Updated legend text) + 
  geom_segment(data = pc_loadings, aes(x = 0, y = 0, xend = PC1 * 2.1, yend = PC2 * 2.1),  # Scale arrows for visibility
               arrow = arrow(length = unit(0.2, "cm")), inherit.aes = FALSE, color = "black", linewidth = 0.8) +
  geom_text(data = pc_loadings, aes(x = PC1 * 2.1, y = PC2 * 2.1, label = Variable,hjust = hjust, vjust = vjust), inherit.aes = FALSE, color = "black", size=2.5) +
  geom_hline(yintercept = 0, linetype = "solid", color = "gray") +  # Add solid horizontal line at y = 0
  geom_vline(xintercept = 0, linetype = "solid", color = "gray") +  # Add solid vertical line at x = 0
  guides(color = guide_legend(title = "Estuary"), shape = guide_legend(title = "Sample Date")) +
  labs(x = paste0("PC1 (", pc1_var_17, "%)"), y = paste0("PC2 (", pc2_var, "%)")) + 
  theme_minimal() +
  theme(
    axis.line = element_line(color = "black"), # Add axis lines
    legend.position = "none"            # Remove grid lines
  )+
  xlim(-3,4)+ #manually added this to make them the same in both years
  ylim(-2,3)#manually added this to make them the same in both years

  
#2016 Version:
result_2016<-pca_result_2016 #manually change for either year 
explained_variance_16 <- result_2016$sdev^2 / sum(result_2016$sdev^2) * 100
pc1_var_16 <- round(explained_variance_16[1], 2)
pc2_var_16 <- round(explained_variance_16[2], 2)

#Label the points the same way as the SNP PCA:
result_2016$Names<-rownames(result_2016$x)
result_2016$Estuary<- sub("_.*", "", result_2016$Names)
result_2016$Season <- sub("^[^_]*_", "", result_2016$Names)

#Preparing for customized version:
pc_scores_16 <- as.data.frame(result_2016$x)  # Principal component scores
pc_scores_16$Estuary <- result_2016$Estuary  # Add Estuary grouping
pc_scores_16$Season <- str_extract(result_2016$Season, "[A-Za-z]+") #redoing to only get the seaosn, not also the year. Also doing this because fill is messed up.
pc_scores_16$Season <- as.factor(pc_scores_16$Season)

pc_loadings_16 <- as.data.frame(result_2016$rotation)  # Variable loadings
pc_loadings_16$Variable <- c("Dissolved Oxygen","Salinity","Temperature") # Add variable names

pc_loadings_16 <- pc_loadings_16 %>%
  mutate(
    vjust = ifelse(PC2 > 0, -1.5, 1.5),  # Adjust vertical position based on arrow direction
    hjust = ifelse(PC1 > 0, 0.3, 0.7)    # Adjust horizontal position based on arrow direction
  ) #allows me to later adjust the position of the labels so they don't overlap with the axes/arrows 

pca_colored_plot_16 <- ggplot(pc_scores_16, aes(x = PC1, y = PC2, color = Estuary, shape = Season)) +
  geom_point(size = 2, stroke = 1.2) +
  scale_color_manual(values = c("Younger" = "#2E2585", "Scott" = "#34B6FF", "Lombardi" = "#EADA02", "OldDairy" = "#D55E00", "Laguna" = "#287762", "Waddell" = "#A000E0")) +
  #scale_shape_manual(values = c("Spring_2016" = 1, "Fall_2016" = 19, "Spring_2017" = 0, "Fall_2017" = 15)) + #old version when all env data on same graph
  scale_shape_manual(values = c("Spring" = 1, "Fall" = 19)) + # Updated legend text) + 
  geom_segment(data = pc_loadings_16, aes(x = 0, y = 0, xend = PC1 * 2.1, yend = PC2 * 2.1),  # Scale arrows for visibility
               arrow = arrow(length = unit(0.2, "cm")), inherit.aes = FALSE, color = "black", linewidth = 0.8) +
  geom_text(data = pc_loadings_16, aes(x = PC1 * 2.1, y = PC2 * 2.1, label = Variable,hjust = hjust, vjust = vjust), inherit.aes = FALSE, color = "black", size=2.5) +
  geom_hline(yintercept = 0, linetype = "solid", color = "gray") +  # Add solid horizontal line at y = 0
  geom_vline(xintercept = 0, linetype = "solid", color = "gray") +  # Add solid vertical line at x = 0
  guides(color = guide_legend(title = "Estuary"), shape = guide_legend(title = "Sample Date")) +
  labs(x = paste0("PC1 (", pc1_var_16, "%)"), y = paste0("PC2 (", pc2_var_16, "%)")) + 
  theme_minimal() +
  theme(
    axis.line = element_line(color = "black"), # Add axis lines
    legend.position = "none"            # Remove legend
  )+
  xlim(-3,4)+ #manually added this to make them the same in both years
  ylim(-2,3)#manually added this to make them the same in both years
  ```

# 3.2 PCA outlier SNPs
*Goal:* Generate PCAs of outliers only

Did the same thing for allele_freq_data_2016_outliers_no_constants.tsv
Only showing for 2017 but same thing done for 2016.
```bash
cut -f1 allele_freq_data_2017_outliers_no_constants.tsv > allele_freq_data_2017_outliers_no_constants_populations.tsv #making a file with population names
```
``` R
library(ggplot2)
library(factoextra)
library(tidyverse)
library(dplyr)
library(data.table)

## 2017 outliers:

#Read the data
file<- "allele_freq_data_2017_outliers_no_constants.tsv"
allele_freq_data <- fread(file, header = TRUE, sep = "\t")

print("data loaded")

#Perform PCA
pca_result <- prcomp(allele_freq_data[, -1], center = TRUE, scale = TRUE) #run pca on everything except text (population names in first column)

#Save pca results
saveRDS(pca_result, file = paste0(sub(".tsv", "", file), "_pca_result.rds"))
write.table(pca_result$sdev, file = paste0(sub(".tsv", "", file), "_pca_result_sd.tsv"))
write.table(pca_result$rotation, file = paste0(sub(".tsv", "", file), "_pca_result_rotation.tsv"))
write.table(pca_result$x, file = paste0(sub(".tsv", "", file), "_pca_result_scores.tsv"))
```


# 3.3 Graphing Outlier PCA and combining with env. PCA
*Goal:* Generate graphical PCA of outlier data, and combine all PCAs for figure. 

``` R
pca_result_outlier_2017<-readRDS("/home/mary2018/links/projects/rrg-barrett/mary2018/MultiYear_FINAL_VERSION/analysis/masked/REDO/poolfstat-fst-version-ii/pca_processing_generation/allele_freq_data_2017_outliers_no_constants_pca_result.rds")
file_outlier_2017<- "/home/mary2018/links/projects/rrg-barrett/mary2018/MultiYear_FINAL_VERSION/analysis/masked/REDO/poolfstat-fst-version-ii/pca_processing_generation/allele_freq_data_2017_outliers_no_constants_populations.tsv"

allele_freq_data_2017 <- fread(file_outlier_2017, header = TRUE, sep = "\t") 
rownames(pca_result_outlier_2017$x) <- allele_freq_data_2017$population

#Process the data
allele_freq_data_2017[, estuary := sapply(strsplit(population, "_"), `[`, 1)]
allele_freq_data_2017[, season := sapply(strsplit(population, "_|-"), `[`, 2)]
allele_freq_data_2017[, season := as.factor(season)]
allele_freq_data_2017[, estuary := as.factor(estuary)]

#Scree plot
scree_plot <- fviz_eig(pca_result_outlier_2017)


#Populations only (coloured by estuary, shape by year)
pc_scores_2017_outlier <- as.data.table(pca_result_outlier_2017$x[, 1:3])
pc_scores_2017_outlier[, Population := allele_freq_data_2017$population]

pc_scores_2017_outlier[, estuary := sub("_.*", "", Population)]
pc_scores_2017_outlier[, SampleDate := sub("^[^_]*_", "", Population)]


explained_variance_outlier_2017 <- pca_result_outlier_2017$sdev^2 / sum(pca_result_outlier_2017$sdev^2) * 100
pc1_var_outlier_2017 <- round(explained_variance_outlier_2017[1], 2)
pc2_var_outlier_2017 <- round(explained_variance_outlier_2017[2], 2)

pc_scores_2017_outlier <- pc_scores_2017_outlier %>%
  mutate(Season = sub("\\..*", "", SampleDate))

pca_colored_plot_2017_outlier <- ggplot(pc_scores_2017_outlier, aes(x = PC1, y = PC2, color = estuary, shape = Season)) +
  geom_point(size = 2, stroke = 1.2) +
  geom_hline(yintercept = 0, linetype = "solid", color = "gray") +  # Add solid horizontal line at y = 0
  geom_vline(xintercept = 0, linetype = "solid", color = "gray") +  # Add solid vertical line at x = 0
  guides(color = guide_legend(title = "Estuary"), shape = guide_legend(title = "Sample Date")) +
  scale_color_manual(values = c("Younger" = "#2E2585", "Scott" = "#34B6FF", "Lombardi" = "#EADA02", "OldDairy" = "#D55E00", "Laguna" = "#287762", "Waddell" = "#A000E0")) +
  scale_shape_manual(values = c("Spring" = 1, "Fall" = 19)) + 
  labs(x = paste0("PC1 (", pc1_var_outlier_2017, "%)"), y = paste0("PC2 (", pc2_var_outlier_2017, "%)")) + 
  theme_minimal() +
  theme(
    axis.line = element_line(color = "black") # Add axis lines
  )

  #For arrows between samples, gathering relevant PCS and arranging in Spring -> Fall order:
  spring_fall_pairs_2017_outliers <- pc_scores_2017_outlier %>%
  filter(Season %in% c("Spring", "Fall")) %>%
  arrange(estuary, factor(Season, levels = c("Spring", "Fall")))

#Create a data frame for the paths
paths_outlier_2017 <- spring_fall_pairs_2017_outliers %>%
  group_by(estuary) %>%
  reframe(
    x = PC1,
    y = PC2,
    Season = Season,
    .groups = 'drop'
  )

#Add the paths to the existing plot
pca_colored_plot_arrows_2017 <- pca_colored_plot_2017_outlier +
  geom_path(data = paths_outlier_2017, aes(x = x, y = y, group = estuary),
            arrow = arrow(length = unit(0.2, "cm")), color = "black")
            
# 2016:
pca_result_outlier_2016<-readRDS("/home/mary2018/links/projects/rrg-barrett/mary2018/MultiYear_FINAL_VERSION/analysis/masked/REDO/poolfstat-fst-version-ii/pca_processing_generation/allele_freq_data_2016_outliers_no_constants_pca_result.rds")
file_outlier_2016<- "/home/mary2018/links/projects/rrg-barrett/mary2018/MultiYear_FINAL_VERSION/analysis/masked/REDO/poolfstat-fst-version-ii/pca_processing_generation/allele_freq_data_2016_outliers_no_constants_populations.tsv"

allele_freq_data_2016 <- fread(file_outlier_2016, header = TRUE, sep = "\t") 
rownames(pca_result_outlier_2016$x) <- allele_freq_data_2016$population

#Process the data
allele_freq_data_2016[, estuary := sapply(strsplit(population, "_"), `[`, 1)]
allele_freq_data_2016[, season := sapply(strsplit(population, "_|-"), `[`, 2)]
allele_freq_data_2016[, season := as.factor(season)]
allele_freq_data_2016[, estuary := as.factor(estuary)]

#Scree plot
scree_plot <- fviz_eig(pca_result_outlier_2016)


#Populations only (coloured by estuary, shape by year)
pc_scores_2016_outlier <- as.data.table(pca_result_outlier_2016$x[, 1:3])
pc_scores_2016_outlier[, Population := allele_freq_data_2016$population]

pc_scores_2016_outlier[, estuary := sub("_.*", "", Population)]
pc_scores_2016_outlier[, SampleDate := sub("^[^_]*_", "", Population)]

explained_variance_outlier_2016 <- pca_result_outlier_2016$sdev^2 / sum(pca_result_outlier_2016$sdev^2) * 100
pc1_var_outlier_2016 <- round(explained_variance_outlier_2016[1], 2)
pc2_var_outlier_2016 <- round(explained_variance_outlier_2016[2], 2)

pc_scores_2016_outlier <- pc_scores_2016_outlier %>%
  mutate(Season = sub("\\..*", "", SampleDate))

pca_colored_plot_2016_outlier <- ggplot(pc_scores_2016_outlier, aes(x = PC1, y = PC2, color = estuary, shape = Season)) +
  geom_point(size = 2, stroke = 1.2) +
  geom_hline(yintercept = 0, linetype = "solid", color = "gray") +  # Add solid horizontal line at y = 0
  geom_vline(xintercept = 0, linetype = "solid", color = "gray") +  # Add solid vertical line at x = 0
  guides(color = guide_legend(title = "Estuary"), shape = guide_legend(title = "Sample Date")) +
  scale_color_manual(values = c("Younger" = "#2E2585", "Scott" = "#34B6FF", "Lombardi" = "#EADA02", "OldDairy" = "#D55E00", "Laguna" = "#287762", "Waddell" = "#A000E0")) +
  scale_shape_manual(values = c("Spring" = 1, "Fall" = 19)) + 
  labs(x = paste0("PC1 (", pc1_var_outlier_2016, "%)"), y = paste0("PC2 (", pc2_var_outlier_2016, "%)")) + 
  theme_minimal() +
  theme(
    axis.line = element_line(color = "black"), # Add axis lines
    legend.position="none" 
  )

  #For arrows between samples, gathering relevant PCS and arranging in Spring -> Fall order:
  spring_fall_pairs_2016_outliers <- pc_scores_2016_outlier %>%
  filter(Season %in% c("Spring", "Fall")) %>%
  arrange(estuary, factor(Season, levels = c("Spring", "Fall")))

#Create a data frame for the paths
paths_outlier_2016 <- spring_fall_pairs_2016_outliers %>%
  group_by(estuary) %>%
  reframe(
    x = PC1,
    y = PC2,
    Season = Season,
    .groups = 'drop'
  )

#Add the paths to the existing plot
pca_colored_plot_arrows_2016 <- pca_colored_plot_2016_outlier +
  geom_path(data = paths_outlier_2016, aes(x = x, y = y, group = estuary),
            arrow = arrow(length = unit(0.2, "cm")), color = "black")

#Now I want to save as 2x2 figure:
combined_pca <- (pca_colored_plot_arrows_2016 + pca_colored_plot_arrows_2017)/
                (pca_colored_plot_16 + pca_colored_plot_17) +
  plot_annotation(tag_levels = 'A') & 
  theme(plot.tag = element_text(size = 14, face = "bold"))

ggsave(filename = paste0("pca_combined.png"), plot = combined_pca, width = 8, height = 7, dpi = 700)
ggsave(filename = paste0("pca_combined.pdf"), plot = combined_pca, width = 8, height = 7)